In [14]:
!pip install py7zr

from ftplib import FTP
import py7zr
import os
import tempfile
import unidecode
import pandas as pd
import numpy as np

ANO = "2026"
ANO_MES = "202602"

FTP_SERVER = "ftp.mtps.gov.br"
FTP_DIRECTORY = f"pdet/microdados/NOVO CAGED/{ANO}/{ANO_MES}/"
LH_FILES_PATH = "/lakehouse/default/Files/"
CODIGO_OSASCO = 353440

StatementMeta(, c479513a-d933-4bac-8e21-5db5d5dcb27b, 16, Finished, Available, Finished, False)

In [15]:
def extrair_ftp(FTP_SERVER, FTP_DIRECTORY, LH_FILES_PATH):

    ftp = FTP(FTP_SERVER, encoding="latin1")
    ftp.login()
    ftp.cwd(FTP_DIRECTORY)
    files = ftp.nlst()

    files_to_download = [file for file in files if file.endswith('.7z')]

    dir_arquivos_extraidos = os.path.join(LH_FILES_PATH, "raw_novo_caged")
    os.makedirs(dir_arquivos_extraidos, exist_ok=True)


    def download_and_extract_7z(file_name, dir_arquivos_extraidos):
        # download
        temp_file = tempfile.NamedTemporaryFile(delete=False)
        local_file_path = temp_file.name
        
        with open(local_file_path, 'wb') as local_file:
            def callback(data):
                local_file.write(data)
            ftp.retrbinary(f"RETR {file_name}", callback)

        extracted_dir = os.path.join(dir_arquivos_extraidos, file_name.split('.')[0])
        os.makedirs(extracted_dir, exist_ok=True)
        
        # extract
        with py7zr.SevenZipFile(local_file_path, mode='r') as z:
            z.extractall(path=extracted_dir)
        
        os.remove(local_file_path)

        print(f"Download concluído: {file_name}")

    def importar_dataframe(pasta_mes, arquivo_mes, dir_arquivos_extraidos):
        PATH_ARQUIVO = os.path.join(dir_arquivos_extraidos, pasta_mes, arquivo_mes)
        df = pd.read_csv(PATH_ARQUIVO, sep=";")
        return df


    for file in files_to_download:
        download_and_extract_7z(file, dir_arquivos_extraidos)
    ftp.quit()

    pasta_mes = {
        f"CAGEDEXC{ANO_MES}": f"CAGEDEXC{ANO_MES}.txt", 
        f"CAGEDFOR{ANO_MES}": f"CAGEDFOR{ANO_MES}.txt",
        f"CAGEDMOV{ANO_MES}": f"CAGEDMOV{ANO_MES}.txt"
    }

    dict_dfs = {}
    for chave, valor in pasta_mes.items():
        dict_dfs[chave] = importar_dataframe(chave, valor, dir_arquivos_extraidos)

    return dict_dfs


def consolidar_caged_exclusoes_fora_prazo(dfs, CODIGO_OSASCO):

    def tratar_nomes_colunas(df):
        df.columns = [unidecode.unidecode(col) for col in df.columns]
        df.columns = [col.lower() for col in df.columns]
        df.columns = [col.replace(" ", "_") for col in df.columns]
        df.columns = [col.replace("-", "_") for col in df.columns]
        df.columns = [col.replace("(", "") for col in df.columns]
        df.columns = [col.replace(")", "") for col in df.columns]
        df.columns = [col.replace(".", "") for col in df.columns]
        df.columns = [col.replace(",", "") for col in df.columns]
        df.columns = [col.replace("'", "") for col in df.columns]
        df.columns = [col.replace("'", "") for col in df.columns]
        return df


    def tratar_caged(df):
        df = df.rename(
            columns={
                "uf": "sigla_uf",
                "uf_desc": "sigla_uf_nome",
                "municipio": "id_municipio",
                "tipoestabelecimento": "tipo_estabelecimento",
                "tipomovimentacao": "tipo_movimentacao_desagregado",
                "horascontratuais": "quantidade_horas_contratadas",
                "salario": "salario_mensal",  # entender se tem conversão - há uma coluna que fala da unidade que está o salario (mensal, anual...)
                "saldomovimentacao": "saldo_movimentacao",
                "indicadoraprendiz": "indicador_aprendiz",
                "indtrabintermitente": "indicador_trabalho_intermitente",
                "indtrabparcial": "indicador_trabalho_parcial",
                "tipodedeficiencia": "tipo_deficiencia",
                "cbo2002ocupacao": "cbo_2002",
                "graudeinstrucao": "grau_instrucao",
                "racacor": "raca_cor",
                "tamestabjan": "tamanho_estabelecimento",
            }
        )

        df["ano"] = df["competenciamov"].astype(str).str[:4].astype(int)
        df["mes"] = df["competenciamov"].astype(str).str[-2:].astype(int)

        df = df.drop(
            [
                "categoria",
                "competenciadec",
                "competenciamov",
                "indicadordeforadoprazo",
                "origemdainformacao",
                "regiao",
                "secao",
                "tipoempregador",
                "unidadesalariocodigo",
                "valorsalariofixo"
            ],
            axis=1,
        )

        return df


    exc = dfs[f'CAGEDEXC{ANO_MES}'].copy()
    fora = dfs[f'CAGEDFOR{ANO_MES}'].copy()
    mov = dfs[f'CAGEDMOV{ANO_MES}'].copy()


    mov = tratar_nomes_colunas(mov)
    fora = tratar_nomes_colunas(fora)
    exc = tratar_nomes_colunas(exc)
    exc = exc.drop(["competenciaexc", "indicadordeexclusao"], axis=1)

    caged_mes_adicional = pd.concat([mov, exc, fora], axis=0, ignore_index=True)

    caged_mes_adicional = tratar_caged(caged_mes_adicional)

    caged_mes_adicional = caged_mes_adicional.loc[caged_mes_adicional['id_municipio'] == CODIGO_OSASCO].copy()

    return caged_mes_adicional


def incluir_descricoes(df):
    dicionario = pd.read_csv("abfss://96fe5a53-3a22-4443-8d0a-e2f6d61a2690@onelake.dfs.fabric.microsoft.com/66918002-484f-4823-bb7f-0e4131dcbd26/Files/aux_tables/br_me_caged_dicionario.csv")

    def aplicar_layout_caged(df):
        """
        Aplica o layout dos dados do CAGED. Colunas com descrições dos códigos.
        Args:
            df (DataFrame): DataFrame com os dados do CAGED.
        Returns:
            df (DataFrame): DataFrame com os dados do CAGED aplicado o layout.
        """
        # leitura dos dados de layout
        excel_layout = pd.ExcelFile("abfss://96fe5a53-3a22-4443-8d0a-e2f6d61a2690@onelake.dfs.fabric.microsoft.com/66918002-484f-4823-bb7f-0e4131dcbd26/Files/aux_tables/Layout Não-identificado Novo Caged Movimentação.xlsx")
        dfs_layout = {}
        for aba in excel_layout.sheet_names:
            dfs_layout[aba] = pd.read_excel(excel_layout, sheet_name=aba)
        # aplicação dos deparas de layout
        for key in dfs_layout.keys():
            if key in df.columns:
                df[f"{key}_desc"] = df[key].map(
                    dfs_layout[key].set_index("Código")["Descrição"]
                )
        df.columns = df.columns.map(unidecode.unidecode)
        return df

    def incluir_descricoes_cnae(df):
        path_cnae = "abfss://96fe5a53-3a22-4443-8d0a-e2f6d61a2690@onelake.dfs.fabric.microsoft.com/66918002-484f-4823-bb7f-0e4131dcbd26/Files/aux_tables/cnae.csv"
        df_cnae = pd.read_csv(path_cnae, encoding="latin1", sep=";")
        df_cnae = df_cnae.rename(
            columns={
                "cnae_20_subclasse": "subclasse",
                "secao_cnae": "secao_cnae_desc",
                "divisao_cnae": "divisao_cnae_desc",
                "grupo_cnae": "grupo_cnae_desc",
            }
        )
        for col in ["secao_cnae_desc", "divisao_cnae_desc", "grupo_cnae_desc"]:
            df_cnae[col] = df_cnae[col].str.capitalize()
        df_join = pd.merge(df, df_cnae, on="subclasse", how="left")
        return df_join

    def incluir_descricao_cbo(df):
        # DESCRIÇÃO CBO 2002
        dicionario_cbo = pd.read_csv("abfss://96fe5a53-3a22-4443-8d0a-e2f6d61a2690@onelake.dfs.fabric.microsoft.com/66918002-484f-4823-bb7f-0e4131dcbd26/Files/aux_tables/br_bd_diretorios_brasil_cbo_2002.csv")
        dicionario_cbo = (
            dicionario_cbo.loc[dicionario_cbo["cbo_2002"].str.isnumeric()][
                ["cbo_2002", "descricao"]
            ]
            .drop_duplicates()
            .rename(columns={"descricao": "cbo_2002_descricao"})
            .astype({"cbo_2002": "int"})
        )
        df = pd.merge(df, dicionario_cbo, on="cbo_2002", how="left")

        return df

    def incluir_secao_cnae(df_caged):
        # SEÇÃO CNAE
        # inclusão da descrição da seção da cnae CAGED
        cnae2 = cnae2 = pd.read_csv("abfss://96fe5a53-3a22-4443-8d0a-e2f6d61a2690@onelake.dfs.fabric.microsoft.com/66918002-484f-4823-bb7f-0e4131dcbd26/Files/aux_tables/br_bd_diretorios_brasil_cnae_2.csv")
        cnae2_subclasse = (
            cnae2[["subclasse", "classe", "descricao_secao"]]
            .drop_duplicates()
            .astype({"subclasse": int})
        )

        df_caged = df_caged.astype({"subclasse": int})
        df_caged = df_caged.merge(cnae2_subclasse, on="subclasse", how="left")
        df_caged = df_caged.rename(
            columns={"descricao_secao": "cnae_2_descricao_secao", "classe": "cnae_2"}
        )
        df_caged = df_caged.drop("subclasse", axis=1)
        return df_caged

    def harmonizar_consolidar_atualizacao(novo_caged):
        # HARMONIZAÇÃO
        novo_caged = novo_caged[
            [
                "ano",
                "mes",
                "sigla_uf",
                "id_municipio",
                "tipo_estabelecimento",
                "tipo_movimentacao_desagregado",
                "quantidade_horas_contratadas",
                "salario_mensal",
                "saldo_movimentacao",
                "indicador_aprendiz",
                "indicador_trabalho_intermitente",
                "indicador_trabalho_parcial",
                "tipo_deficiencia",
                "cbo_2002",
                "cbo_2002_descricao",
                "cnae_2",
                "grau_instrucao",
                "idade",
                "sexo",
                "raca_cor",
                "tamanho_estabelecimento",
                "cnae_2_descricao_secao",
            ]
        ].copy()

        cols_valores = ["quantidade_horas_contratadas", "salario_mensal"]
        for col in cols_valores:
            novo_caged[col] = novo_caged[col].str.replace(",", ".").astype(float).fillna(0)

        # AJUSTE FINAL DE TIPOS E CONCATENAÇÃO
        tipos_novo_caged = {
            "ano": "int32",
            "mes": "int32",
            "sigla_uf": str,
            "id_municipio": "int32",
            "tipo_estabelecimento": "int32",
            "tipo_movimentacao_desagregado": "int32",
            "quantidade_horas_contratadas": "int32",
            "salario_mensal": float,
            "saldo_movimentacao": "int32",
            "indicador_aprendiz": "int32",
            "indicador_trabalho_intermitente": float,
            "indicador_trabalho_parcial": float,
            "tipo_deficiencia": float,
            "cbo_2002": str,
            "cbo_2002_descricao": str,
            "cnae_2": "int32",
            "grau_instrucao": "int32",
            "idade": "int32",
            "sexo": "int32",
            "raca_cor": float,
            "tamanho_estabelecimento": float,
            "cnae_2_descricao_secao": str,
        }
        novo_caged = novo_caged.astype(tipos_novo_caged)

        novo_caged['sigla_uf_nome'] = "São Paulo"
        novo_caged['id_municipio_nome'] = "Osasco"

        novo_caged["id_tabela"] = np.where(
            novo_caged["ano"] <= 2019, "microdados_antigos", "microdados_movimentacao"
        )

        return novo_caged

    def incluir_descricao_raca_cor(caged_total, dicionario):
        caged_total["raca_cor"] = caged_total["raca_cor"].fillna(99)
        caged_total = caged_total.astype({"raca_cor": int})
        dicionario_raca = (
            dicionario.loc[dicionario["nome_coluna"] == "raca_cor"]
            .rename(columns={"chave": "raca_cor", "valor": "raca_cor_descricao"})
            .drop(["nome_coluna", "cobertura_temporal"], axis=1)
        )
        caged_total = pd.merge(
            caged_total, dicionario_raca, on=["id_tabela", "raca_cor"], how="left"
        )
        return caged_total

    def incluir_descricao_sexo(caged_total):
        dict_sexo = {
            1: "Masculino",
            2: "Feminino",
            3: "Feminino",
            9: "Não Identificado",
            -1: "Ignorado",
        }
        caged_total["sexo_descricao"] = caged_total["sexo"].map(dict_sexo)

        return caged_total

    def incluir_descricao_grau_instrucao(caged_total, dicionario):
        dicionario_instrucao = (
            dicionario.loc[dicionario["nome_coluna"] == "grau_instrucao"]
            .rename(
                columns={"chave": "grau_instrucao", "valor": "grau_instrucao_descricao"}
            )
            .drop(["nome_coluna", "cobertura_temporal"], axis=1)
        )
        caged_total = pd.merge(
            caged_total,
            dicionario_instrucao,
            on=["id_tabela", "grau_instrucao"],
            how="left",
        )
        return caged_total

    def incluir_descricao_deficiencia(caged_total, dicionario):
        dicionario_deficiencia = (
            dicionario.loc[dicionario["nome_coluna"] == "tipo_deficiencia"]
            .rename(
                columns={"chave": "tipo_deficiencia", "valor": "tipo_deficiencia_descricao"}
            )
            .drop(["nome_coluna", "cobertura_temporal"], axis=1)
        )
        dicionario_deficiencia["tipo_deficiencia_descricao"] = dicionario_deficiencia[
            "tipo_deficiencia_descricao"
        ].replace(
            {
                "Física": "Fisica",
                "Múltipla": "Multipla",
                "Nao Defic": "Não Deficiente",
            }
        )
        caged_total = pd.merge(
            caged_total,
            dicionario_deficiencia,
            on=["id_tabela", "tipo_deficiencia"],
            how="left",
        )
        return caged_total


    caged_mes_adicional = aplicar_layout_caged(df)
    caged_mes_adicional = incluir_descricoes_cnae(caged_mes_adicional)
    caged_mes_adicional = incluir_descricao_cbo(caged_mes_adicional)
    caged_mes_adicional = incluir_secao_cnae(caged_mes_adicional)
    caged_mes_adicional = harmonizar_consolidar_atualizacao(caged_mes_adicional)

    caged_mes_adicional = incluir_descricao_raca_cor(caged_mes_adicional, dicionario)
    caged_mes_adicional = incluir_descricao_sexo(caged_mes_adicional)
    caged_mes_adicional = incluir_descricao_grau_instrucao(caged_mes_adicional, dicionario)
    caged_mes_adicional = incluir_descricao_deficiencia(caged_mes_adicional, dicionario)

    return caged_mes_adicional


StatementMeta(, c479513a-d933-4bac-8e21-5db5d5dcb27b, 17, Finished, Available, Finished, False)

In [16]:
dfs = extrair_ftp(FTP_SERVER, FTP_DIRECTORY, LH_FILES_PATH)
caged_mes_adicional = consolidar_caged_exclusoes_fora_prazo(dfs, CODIGO_OSASCO)
caged_mes_adicional = incluir_descricoes(caged_mes_adicional)

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, ShortType, ByteType, LongType

def ints_to_long(df):
    cols = []
    for f in df.schema.fields:
        if isinstance(f.dataType, (IntegerType, ShortType, ByteType)):
            cols.append(F.col(f.name).cast(LongType()).alias(f.name))  # "long" = "bigint"
        else:
            cols.append(F.col(f.name))
    return df.select(*cols)

df = spark.createDataFrame(caged_mes_adicional)
df = ints_to_long(df)

df.write.mode("append").format("delta").saveAsTable("silver_caged")

StatementMeta(, c479513a-d933-4bac-8e21-5db5d5dcb27b, 18, Finished, Available, Finished, True)

Download concluído: CAGEDEXC202602.7z
Download concluído: CAGEDFOR202602.7z
Download concluído: CAGEDMOV202602.7z
